# GAT-v4 | Phase 3+4 — CNV Processing & Cross-Validation Splits

**Covers plan sections 5 (CNV) and 6 (CV splits)**

Both phases are short (~1 day each in the plan). Combined here for efficiency.

### Phase 3 — CNV Processing
- Restrict CNV matrix to DEG gene symbols
- Assign CNV = 0 for normal samples (diploid assumption)
- Save `cnv_aligned.csv` — centering on training tumor mean happens **per-fold** in Phase 6 (feature tensor)

### Phase 4 — CV Splits
- 5-fold × 3 seeds = **15 total fits**
- Stratified on **event × stage** combined label
- Splits at **patient level** — a patient never appears in both train and test
- Within each training set: 90% train / 10% val (stratified)
- Output: `cv_folds_repeated_5x3.json`

---
| Cell | What |
|------|------|
| 1 | Imports & paths |
| 2 | Load DEG list + CNV raw |
| 3 | Align CNV to DEG symbols |
| 4 | Add normal samples (CNV=0) |
| 5 | Save cnv_aligned.csv |
| 6 | Load clinical for CV splits |
| 7 | Build stratification label |
| 8 | Generate 5×3 CV folds |
| 9 | Validate splits |
| 10 | Save splits + audit YAML |

---
## Cell 1 — Imports & Paths

In [16]:
import sys, json, yaml, hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
AUDIT_DIR    = KIRC_ROOT / "audit"
# ─────────────────────────────────────────────────────────────────────────────

DATA_SPLITS.mkdir(parents=True, exist_ok=True)

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import CV_REPEAT_SEEDS, N_CV_FOLDS, MASTER_SEED

log = get_logger('phase3_cnv_splits')

print(f"CV seeds   : {CV_REPEAT_SEEDS}")
print(f"CV folds   : {N_CV_FOLDS}")
print(f"Total fits : {len(CV_REPEAT_SEEDS) * N_CV_FOLDS}")
print("Imports OK.")

2026-07-06 22:41:09 | INFO     | phase3_cnv_splits | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase3_cnv_splits_20260706_224109.log


CV seeds   : [42, 123, 456]
CV folds   : 5
Total fits : 15
Imports OK.


---
## Cell 2 — Load DEG List & CNV Raw

We use `deg_results.csv` to get the exact DEG symbols used in the pipeline,
and `cnv_raw.csv` which was saved in Phase 1 (tumor patients only).

In [17]:
# ── DEG symbols ───────────────────────────────────────────────────────────────
deg_res = pd.read_csv(DATA_PROC / 'deg_results.csv', index_col=0)

# DEGs = genes passing padj < 0.05 AND |LFC| > 1.0
deg_mask = (
    deg_res['padj'].notna() &
    (deg_res['padj'] < 0.05) &
    (deg_res['log2FoldChange'].abs() > 1.0)
)
deg_symbols = deg_res[deg_mask].index.tolist()
deg_set     = set(deg_symbols)

print(f"DEG symbols loaded : {len(deg_symbols):,}")
print(f"Examples           : {deg_symbols[:8]}")

# ── CNV raw ───────────────────────────────────────────────────────────────────
print("\nLoading cnv_raw.csv...")
cnv_raw = pd.read_csv(DATA_PROC / 'cnv_raw.csv', index_col=0)

print(f"CNV raw shape      : {cnv_raw.shape}  (genes x tumor_samples)")
print(f"CNV gene examples  : {list(cnv_raw.index[:5])}")
print(f"CNV col examples   : {list(cnv_raw.columns[:3])}")
print(f"GISTIC values      : {sorted(cnv_raw.stack().dropna().astype(int).unique().tolist())}")

# ── Sample alignment ─────────────────────────────────────────────────────────
align = pd.read_csv(DATA_PROC / 'sample_alignment.csv')
normal_sids  = align[(align['tissue_type']=='normal') & align['include']]['sample_id'].tolist()
tumor_sids   = align[(align['tissue_type']=='tumor')  & align['include']]['sample_id'].tolist()
print(f"\nAlignment: {len(tumor_sids)} tumor, {len(normal_sids)} normal included")

log.info(f"DEGs: {len(deg_symbols)}; CNV raw: {cnv_raw.shape}")
print("\nCell 2 done.")

DEG symbols loaded : 9,997
Examples           : ['TSPAN6', 'FGR', 'SEMA3F', 'CYP51A1', 'CD99', 'HECW1', 'MAD1L1', 'TMEM176A']

Loading cnv_raw.csv...
CNV raw shape      : (24776, 491)  (genes x tumor_samples)
CNV gene examples  : ['ACAP3', 'ACTRT2', 'AGRN', 'ANKRD65', 'ATAD3A']
CNV col examples   : ['TCGA-3Z-A93Z-01', 'TCGA-6D-AA2E-01', 'TCGA-A3-3306-01']


2026-07-06 22:41:12 | INFO     | phase3_cnv_splits | DEGs: 9997; CNV raw: (24776, 491)


GISTIC values      : [-2, -1, 0, 1, 2]

Alignment: 500 tumor, 69 normal included

Cell 2 done.


---
## Cell 3 — Align CNV to DEG Symbols

Keep only genes that are in both the CNV matrix and the DEG list.

Expected overlap: ~60–80% of DEGs have CNV data (some genes not in GISTIC output).
Genes absent from CNV are assigned CNV = 0 (no detected copy number alteration — conservative).

In [18]:
# Cell 3 — Align CNV to DEG Symbols (FIXED v2)

# ── Step 1: deduplicate deg_symbols ──────────────────────────────────────────
n_deg_dup = len(deg_symbols) - len(set(deg_symbols))
print(f"Duplicate symbols in DEG list : {n_deg_dup}")
if n_deg_dup > 0:
    # Keep order, drop later duplicates
    seen = set()
    deg_symbols_u = []
    for g in deg_symbols:
        if g not in seen:
            deg_symbols_u.append(g)
            seen.add(g)
    print(f"  DEG symbols after dedup : {len(deg_symbols_u):,}")
else:
    deg_symbols_u = deg_symbols

# ── Step 2: deduplicate CNV raw index ────────────────────────────────────────
n_cnv_dup = cnv_raw.index.duplicated().sum()
print(f"Duplicate symbols in CNV raw  : {n_cnv_dup}")
if n_cnv_dup > 0:
    cnv_dedup = (
        cnv_raw
        .assign(_abs_mean=cnv_raw.abs().mean(axis=1))
        .sort_values('_abs_mean', ascending=False)
        .drop(columns='_abs_mean')
    )
    cnv_dedup = cnv_dedup[~cnv_dedup.index.duplicated(keep='first')]
    print(f"  CNV genes after dedup : {cnv_dedup.shape[0]:,}")
else:
    cnv_dedup = cnv_raw.copy()

# ── Step 3: overlap stats ────────────────────────────────────────────────────
cnv_gene_set = set(cnv_dedup.index)
deg_in_cnv   = [g for g in deg_symbols_u if g in cnv_gene_set]
deg_not_cnv  = [g for g in deg_symbols_u if g not in cnv_gene_set]
overlap_pct  = len(deg_in_cnv) / len(deg_symbols_u) * 100

print(f"\nDEG symbols (unique)       : {len(deg_symbols_u):,}")
print(f"DEGs with CNV data         : {len(deg_in_cnv):,}  ({overlap_pct:.1f}%)")
print(f"DEGs absent from CNV → 0   : {len(deg_not_cnv):,}  ({100-overlap_pct:.1f}%)")

# ── Step 4: build aligned matrix ────────────────────────────────────────────
cnv_overlap = cnv_dedup.loc[deg_in_cnv].copy()

if deg_not_cnv:
    zero_rows = pd.DataFrame(
        0.0,
        index   = deg_not_cnv,
        columns = cnv_dedup.columns
    )
    cnv_all_degs = pd.concat([cnv_overlap, zero_rows], axis=0)
else:
    cnv_all_degs = cnv_overlap

# ── Step 5: final dedup safety net then reindex ──────────────────────────────
if cnv_all_degs.index.duplicated().sum() > 0:
    print(f"  Safety dedup on concatenated frame...")
    cnv_all_degs = cnv_all_degs[~cnv_all_degs.index.duplicated(keep='first')]

cnv_all_degs = cnv_all_degs.reindex(deg_symbols_u).fillna(0.0)

# Update deg_symbols to the deduplicated version for all downstream cells
deg_symbols = deg_symbols_u
print(f"\nCNV aligned shape : {cnv_all_degs.shape}  (genes x tumor_samples)")
print(f"Genes == DEG count: {len(cnv_all_degs) == len(deg_symbols)}")

val_counts = cnv_all_degs.stack().astype(int).value_counts().sort_index()
print(f"\nGISTIC value distribution:")
for val, cnt in val_counts.items():
    print(f"  {val:+d}  :  {cnt:,}  ({cnt/val_counts.sum()*100:.1f}%)")

log.info(f"CNV aligned: {cnv_all_degs.shape}")
print("\nCell 3 done.")

Duplicate symbols in DEG list : 0
Duplicate symbols in CNV raw  : 12
  CNV genes after dedup : 24,764

DEG symbols (unique)       : 9,997
DEGs with CNV data         : 5,688  (56.9%)
DEGs absent from CNV → 0   : 4,309  (43.1%)

CNV aligned shape : (9997, 491)  (genes x tumor_samples)
Genes == DEG count: True


2026-07-06 22:41:13 | INFO     | phase3_cnv_splits | CNV aligned: (9997, 491)



GISTIC value distribution:
  -2  :  8,697  (0.2%)
  -1  :  316,291  (6.4%)
  +0  :  4,242,185  (86.4%)
  +1  :  322,316  (6.6%)
  +2  :  19,038  (0.4%)

Cell 3 done.


---
## Cell 4 — Add Normal Samples (CNV = 0)

Normal samples are assumed diploid — CNV = 0 for all genes.
This is standard practice (Love et al. 2014; TCGA KIRC paper).
Mentioned in manuscript limitations: *"Solid tissue normals may carry germline CNVs;
the diploid assumption is standard but introduces a small systematic bias."*

In [19]:
# Normal samples: all get CNV = 0
normal_cnv = pd.DataFrame(
    0.0,
    index   = deg_symbols,
    columns = normal_sids
)

# Combine: tumor CNV (aligned) + normal CNV (zeros)
# Column order: tumor first, normal second (consistent with VST matrix)
# Note: CNV column names are SHORT barcodes (TCGA-XX-YYYY-ZZ)
# while normal_sids are FULL barcodes — kept separate, joined by patient_id at feature time
cnv_aligned = pd.concat([cnv_all_degs, normal_cnv], axis=1)

print(f"CNV aligned shape (tumor + normal) : {cnv_aligned.shape}")
print(f"Tumor columns                      : {cnv_all_degs.shape[1]}")
print(f"Normal columns (all zero)          : {normal_cnv.shape[1]}")
print()

# Verify no NaNs
nan_count = cnv_aligned.isna().sum().sum()
print(f"NaN values : {nan_count}  (expected 0)")
if nan_count > 0:
    print(f"  Filling remaining NaN with 0.")
    cnv_aligned = cnv_aligned.fillna(0.0)

# NOTE: Per-fold centering on training tumor mean is NOT done here.
# Centering is fold-aware and happens in Phase 6 (feature tensor assembly)
# to prevent data leakage from test/val samples into the mean estimate.
print("NOTE: Centering on training tumor mean is fold-aware.")
print("      It will be applied per-fold in Phase 6 (feature tensor assembly).")
print("      Saving raw aligned values here.")

log.info(f"CNV with normals: {cnv_aligned.shape}")
print("\nCell 4 done.")

2026-07-06 22:41:13 | INFO     | phase3_cnv_splits | CNV with normals: (9997, 560)


CNV aligned shape (tumor + normal) : (9997, 560)
Tumor columns                      : 491
Normal columns (all zero)          : 69

NaN values : 0  (expected 0)
NOTE: Centering on training tumor mean is fold-aware.
      It will be applied per-fold in Phase 6 (feature tensor assembly).
      Saving raw aligned values here.

Cell 4 done.


---
## Cell 5 — Save cnv_aligned.csv

In [20]:
import time

cnv_path = DATA_PROC / 'cnv_aligned.csv'
print(f"Saving cnv_aligned.csv  {cnv_aligned.shape} ...", end='', flush=True)
t0 = time.time()
cnv_aligned.to_csv(cnv_path)
print(f" {time.time()-t0:.0f}s  ({cnv_path.stat().st_size/1024**2:.1f} MB)")

Saving cnv_aligned.csv  (9997, 560) ... 6s  (21.8 MB)


---
## Cell 6 — Load Clinical for CV Splits

CV splits use **tumor patients only** (the cohort we train and evaluate on).
Normal samples follow their patient's fold assignment (used in DESeq2, not training).

In [21]:
clin = pd.read_csv(DATA_PROC / 'clinical_clean.csv')
print(f"Clinical shape : {clin.shape}")
print(f"Columns        : {list(clin.columns)}")
print()

# We only split patients in the tumor-included set
align_tumor = align[(align['tissue_type']=='tumor') & align['include']]
tumor_patient_ids = align_tumor['patient_id'].unique().tolist()

# Intersect with clinical (must have survival data)
clin_tumor = clin[clin['patient_id'].isin(tumor_patient_ids)].copy()
clin_tumor = clin_tumor.drop_duplicates('patient_id').reset_index(drop=True)

print(f"Tumor patients in alignment   : {len(tumor_patient_ids)}")
print(f"Tumor patients with clinical  : {len(clin_tumor)}")

missing_clin = set(tumor_patient_ids) - set(clin_tumor['patient_id'])
if missing_clin:
    print(f"\nPatients in alignment but not clinical ({len(missing_clin)}):")
    print(f"  {list(missing_clin)[:5]}")
    print("  These will not appear in any fold (excluded).")

print(f"\nEvent distribution:")
print(f"  Deceased (1): {int(clin_tumor['event'].sum())}")
print(f"  Censored (0): {int((clin_tumor['event']==0).sum())}")
print(f"  Event rate  : {clin_tumor['event'].mean():.1%}")

log.info(f"Clinical for CV: {len(clin_tumor)} tumor patients")
print("\nCell 6 done.")

2026-07-06 22:41:19 | INFO     | phase3_cnv_splits | Clinical for CV: 491 tumor patients


Clinical shape : (532, 6)
Columns        : ['patient_id', 'event', 'os_months', 'age', 'stage', 'grade']

Tumor patients in alignment   : 491
Tumor patients with clinical  : 491

Event distribution:
  Deceased (1): 166
  Censored (0): 325
  Event rate  : 33.8%

Cell 6 done.


---
## Cell 7 — Build Stratification Label

Stratify on **event × stage** combined label so each fold has similar proportions of:
- Deceased vs censored
- Stage I/II vs III/IV

This is the same approach as the plan's `strat_label` in Phase 6.

In [22]:
patients = clin_tumor['patient_id'].values
events   = clin_tumor['event'].fillna(0).astype(int).values

# Stage: use if available, else fall back to event-only stratification
if 'stage' in clin_tumor.columns:
    stages = clin_tumor['stage'].fillna(clin_tumor['stage'].median()).astype(int).values
    # Bin stage: I-II = 'early', III-IV = 'late'
    stage_bin = np.where(stages <= 2, 'early', 'late')
    strat_label = np.array([f"{e}_{s}" for e, s in zip(events, stage_bin)])
    print("Stratification: event × stage_bin (early/late)")
else:
    strat_label = events.astype(str)
    print("Stratification: event only (stage column not found)")

print(f"\nStratification label counts:")
label_counts = Counter(strat_label)
for label, cnt in sorted(label_counts.items()):
    print(f"  '{label}'  :  {cnt} patients  ({cnt/len(patients):.1%})")

# Check minimum class size — StratifiedKFold needs >= n_splits per class
min_class = min(label_counts.values())
print(f"\nSmallest class : {min_class} patients")
if min_class < N_CV_FOLDS:
    print(f"WARNING: smallest class ({min_class}) < n_folds ({N_CV_FOLDS}).")
    print("  Falling back to event-only stratification.")
    strat_label = events.astype(str)
    label_counts = Counter(strat_label)
    print(f"  New label counts: {dict(sorted(label_counts.items()))}")
else:
    print(f"  OK — all classes have >= {N_CV_FOLDS} samples.")

print("\nCell 7 done.")

Stratification: event × stage_bin (early/late)

Stratification label counts:
  '0_early'  :  245 patients  (49.9%)
  '0_late'  :  80 patients  (16.3%)
  '1_early'  :  54 patients  (11.0%)
  '1_late'  :  112 patients  (22.8%)

Smallest class : 54 patients
  OK — all classes have >= 5 samples.

Cell 7 done.


---
## Cell 8 — Generate 5×3 CV Folds

**Structure:** 3 repeats × 5 folds = 15 total fits

Each fold:
- `train_patients`: 90% of training set (used for model fitting)
- `val_patients`: 10% of training set (used for early stopping / hyperparameter selection)
- `test_patients`: held-out fold (used for reporting C-index)

**Key constraint:** splits are at patient level. A patient never has their
tumor and normal samples in different folds.

In [23]:
cv_splits = {}
fold_stats = []

for seed in CV_REPEAT_SEEDS:
    skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=seed)

    for fold_idx, (trainval_idx, test_idx) in enumerate(
        skf.split(patients, strat_label)
    ):
        # Split trainval → train (90%) + val (10%)
        trainval_labels = strat_label[trainval_idx]

        # Check min class size for inner split
        inner_min = min(Counter(trainval_labels).values())
        if inner_min < 2:
            # Fall back to non-stratified if a class has only 1 sample
            sub_train_idx, sub_val_idx = train_test_split(
                trainval_idx,
                test_size=0.10,
                random_state=seed
            )
        else:
            sub_train_idx, sub_val_idx = train_test_split(
                trainval_idx,
                test_size=0.10,
                stratify=trainval_labels,
                random_state=seed
            )

        train_pats = patients[sub_train_idx].tolist()
        val_pats   = patients[sub_val_idx].tolist()
        test_pats  = patients[test_idx].tolist()

        fold_key = f"seed{seed}_fold{fold_idx}"
        cv_splits[fold_key] = {
            'seed'            : int(seed),
            'fold'            : int(fold_idx),
            'train_patients'  : train_pats,
            'val_patients'    : val_pats,
            'test_patients'   : test_pats,
        }

        # Stats for this fold
        train_ev = clin_tumor[clin_tumor['patient_id'].isin(train_pats)]['event'].mean()
        val_ev   = clin_tumor[clin_tumor['patient_id'].isin(val_pats)]['event'].mean()
        test_ev  = clin_tumor[clin_tumor['patient_id'].isin(test_pats)]['event'].mean()

        fold_stats.append({
            'fold_key'      : fold_key,
            'n_train'       : len(train_pats),
            'n_val'         : len(val_pats),
            'n_test'        : len(test_pats),
            'train_ev_rate' : round(train_ev, 3),
            'val_ev_rate'   : round(val_ev,   3),
            'test_ev_rate'  : round(test_ev,  3),
        })

stats_df = pd.DataFrame(fold_stats)

print(f"Total folds generated : {len(cv_splits)}  (expected {N_CV_FOLDS * len(CV_REPEAT_SEEDS)})")
print()
print("Fold-level statistics:")
print(stats_df.to_string(index=False))
print()
print("Summary across all folds:")
print(f"  Mean train size : {stats_df['n_train'].mean():.0f}  (±{stats_df['n_train'].std():.1f})")
print(f"  Mean val size   : {stats_df['n_val'].mean():.0f}  (±{stats_df['n_val'].std():.1f})")
print(f"  Mean test size  : {stats_df['n_test'].mean():.0f}  (±{stats_df['n_test'].std():.1f})")
print(f"  Mean test event rate : {stats_df['test_ev_rate'].mean():.3f}  (±{stats_df['test_ev_rate'].std():.3f})")
print(f"  Overall event rate   : {clin_tumor['event'].mean():.3f}  (should be close to above)")

log.info(f"CV splits generated: {len(cv_splits)} folds")
print("\nCell 8 done.")

2026-07-06 22:41:19 | INFO     | phase3_cnv_splits | CV splits generated: 15 folds


Total folds generated : 15  (expected 15)

Fold-level statistics:
     fold_key  n_train  n_val  n_test  train_ev_rate  val_ev_rate  test_ev_rate
 seed42_fold0      352     40      99          0.338        0.325         0.343
 seed42_fold1      353     40      98          0.340        0.325         0.337
 seed42_fold2      353     40      98          0.340        0.325         0.337
 seed42_fold3      353     40      98          0.340        0.325         0.337
 seed42_fold4      353     40      98          0.340        0.325         0.337
seed123_fold0      352     40      99          0.338        0.325         0.343
seed123_fold1      353     40      98          0.340        0.325         0.337
seed123_fold2      353     40      98          0.340        0.325         0.337
seed123_fold3      353     40      98          0.340        0.325         0.337
seed123_fold4      353     40      98          0.340        0.325         0.337
seed456_fold0      352     40      99          0.338  

---
## Cell 9 — Validate Splits

Critical checks before saving:
1. No patient appears in both train and test in the same fold
2. No patient appears in both val and test in the same fold
3. All patients appear in exactly 1 test fold per seed
4. Event rates are balanced across folds (stratification working)

In [24]:
checks = {}
all_patients_set = set(patients)

# Check 1 & 2: no leakage between train/val/test within each fold
leakage_found = False
for key, fold in cv_splits.items():
    tr = set(fold['train_patients'])
    va = set(fold['val_patients'])
    te = set(fold['test_patients'])

    tr_te = tr & te
    va_te = va & te
    tr_va = tr & va

    if tr_te or va_te or tr_va:
        print(f"LEAK in {key}: train∩test={len(tr_te)}, val∩test={len(va_te)}, train∩val={len(tr_va)}")
        leakage_found = True

checks['no_train_test_leakage'] = not leakage_found
print(f"  {'OK  ' if not leakage_found else 'FAIL'} No train/val/test leakage in any fold")

# Check 3: each patient appears in exactly 1 test fold per seed
coverage_ok = True
for seed in CV_REPEAT_SEEDS:
    seed_folds = {k: v for k, v in cv_splits.items() if v['seed'] == seed}
    test_counts = Counter()
    for fold in seed_folds.values():
        for p in fold['test_patients']:
            test_counts[p] += 1

    appeared_once   = sum(1 for c in test_counts.values() if c == 1)
    appeared_multi  = sum(1 for c in test_counts.values() if c > 1)
    missing_from_test = len(all_patients_set) - len(test_counts)

    if appeared_multi > 0 or missing_from_test > 0:
        print(f"  Seed {seed}: {appeared_multi} in test >1×, {missing_from_test} never in test")
        coverage_ok = False

checks['each_patient_in_test_exactly_once_per_seed'] = coverage_ok
print(f"  {'OK  ' if coverage_ok else 'FAIL'} Each patient in test exactly once per seed")

# Check 4: event rate variance across test folds
test_ev_rates = stats_df['test_ev_rate'].values
ev_std = test_ev_rates.std()
ev_ok  = ev_std < 0.08   # <8pp std is acceptable for stratified splits
checks['test_event_rate_balanced'] = ev_ok
print(f"  {'OK  ' if ev_ok else 'FAIL'} Test event rate std = {ev_std:.3f}  (threshold <0.08)")

# Check 5: total patients covered
n_covered = len(test_counts)   # from last seed, representative
n_expect  = len(patients)
checks['all_patients_covered'] = n_covered == n_expect
print(f"  {'OK  ' if n_covered == n_expect else 'FAIL'} All {n_expect} patients covered in splits ({n_covered} found)")

# Check 6: normal samples follow tumor patient splits (documented note)
print()
print("NOTE: Normal sample assignment (per plan section 6):")
print("  Normal samples whose patient is in the tumor cohort → follow patient's split.")
print("  Normal samples with no matching tumor patient → assigned to train of every fold.")
print("  This is handled at data-loading time in Phase 7 (feature tensor assembly).")
print("  No action needed here.")

print()
n_pass = sum(checks.values())
print(f"Validation: {n_pass}/{len(checks)} checks passed")
if n_pass < len(checks):
    print("FAILED checks — do not proceed to Phase 5 until resolved.")
else:
    print("All checks passed. Splits are valid.")

print("\nCell 9 done.")

  OK   No train/val/test leakage in any fold
  OK   Each patient in test exactly once per seed
  OK   Test event rate std = 0.002  (threshold <0.08)
  OK   All 491 patients covered in splits (491 found)

NOTE: Normal sample assignment (per plan section 6):
  Normal samples whose patient is in the tumor cohort → follow patient's split.
  Normal samples with no matching tumor patient → assigned to train of every fold.
  This is handled at data-loading time in Phase 7 (feature tensor assembly).
  No action needed here.

Validation: 4/4 checks passed
All checks passed. Splits are valid.

Cell 9 done.


---
## Cell 10 — Save Splits + Audit YAML

In [25]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while data := f.read(8192): h.update(data)
    return h.hexdigest()

# ── Save CV folds ─────────────────────────────────────────────────────────────
splits_path = DATA_SPLITS / 'cv_folds_repeated_5x3.json'
with open(splits_path, 'w') as f:
    json.dump(cv_splits, f, indent=2)
print(f"CV folds saved -> {splits_path}")
print(f"  File size : {splits_path.stat().st_size/1024:.0f} KB")

# Also save fold stats as CSV for quick reference
stats_path = DATA_SPLITS / 'cv_fold_stats.csv'
stats_df.to_csv(stats_path, index=False)
print(f"Fold stats -> {stats_path}")

# ── Audit YAML ────────────────────────────────────────────────────────────────
audit = {
    'phase'     : 'Phase 3+4 — CNV Processing & CV Splits',
    'timestamp' : datetime.now().isoformat(),

    'cnv_processing': {
        'deg_symbols_total'     : len(deg_symbols),
        'degs_with_cnv_data'    : len(deg_in_cnv),
        'degs_without_cnv_data' : len(deg_not_cnv),
        'overlap_pct'           : round(overlap_pct, 1),
        'no_cnv_treatment'      : 'assigned CNV=0 (no detected CNA)',
        'normal_cnv_treatment'  : 'diploid assumption CNV=0 (standard practice)',
        'centering_note'        : 'Per-fold centering on training tumor mean applied in Phase 6 (feature tensor)',
        'output'                : str(cnv_path),
        'output_shape'          : list(cnv_aligned.shape),
    },

    'cv_splits': {
        'n_patients'          : int(len(patients)),
        'n_folds'             : N_CV_FOLDS,
        'n_seeds'             : len(CV_REPEAT_SEEDS),
        'seeds'               : CV_REPEAT_SEEDS,
        'total_fits'          : len(cv_splits),
        'split_level'         : 'patient (not sample)',
        'stratification'      : 'event x stage_bin (early=I-II, late=III-IV)',
        'val_fraction'        : 0.10,
        'mean_train_size'     : round(float(stats_df['n_train'].mean()), 1),
        'mean_val_size'       : round(float(stats_df['n_val'].mean()), 1),
        'mean_test_size'      : round(float(stats_df['n_test'].mean()), 1),
        'test_ev_rate_mean'   : round(float(stats_df['test_ev_rate'].mean()), 3),
        'test_ev_rate_std'    : round(float(stats_df['test_ev_rate'].std()), 3),
        'validation_checks'   : {k: bool(v) for k, v in checks.items()},
        'all_checks_passed'   : bool(n_pass == len(checks)),
        'note_normals'        : 'Normal samples follow patient split assignment; handled in Phase 6 loader',
    },

    'output_checksums': {
        'cnv_aligned.csv'            : file_md5(cnv_path),
        'cv_folds_repeated_5x3.json' : file_md5(splits_path),
    },

    'next_phase' : 'Phase 5 — Graph Construction (05_ppi_construction.ipynb)',
    'status'     : 'COMPLETE' if n_pass == len(checks) else 'REVIEW_NEEDED',
}

audit_path = AUDIT_DIR / 'phase3_4_cnv_splits.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 3+4 COMPLETE")
print("=" * 55)
print(f"  CNV aligned       : {cnv_aligned.shape}")
print(f"  DEG-CNV overlap   : {overlap_pct:.1f}%")
print(f"  CV folds          : {len(cv_splits)} total ({N_CV_FOLDS} folds × {len(CV_REPEAT_SEEDS)} seeds)")
print(f"  Patients per test : ~{stats_df['n_test'].mean():.0f}")
print(f"  Validation        : {n_pass}/{len(checks)} checks passed")
print()
print("Next: 05_ppi_construction.ipynb — STRING PPI subgraph + Pearson edge weights")

CV folds saved -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\splits\cv_folds_repeated_5x3.json
  File size : 168 KB
Fold stats -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\splits\cv_fold_stats.csv
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase3_4_cnv_splits.yaml

PHASE 3+4 COMPLETE
  CNV aligned       : (9997, 560)
  DEG-CNV overlap   : 56.9%
  CV folds          : 15 total (5 folds × 3 seeds)
  Patients per test : ~98
  Validation        : 4/4 checks passed

Next: 05_ppi_construction.ipynb — STRING PPI subgraph + Pearson edge weights
